# MarketLens AI — Phase 0 EDA

Exploratory data analysis and data-quality decisions.

- Source CSVs in `data/raw/` were **not modified**
- Customer names excluded from charts (privacy)
- Run `python scripts/profile_data.py` for machine-readable profile JSON

In [ ]:
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")

orders_raw = pd.read_csv(RAW / "List of Orders.csv")
details = pd.read_csv(RAW / "Order Details.csv")
targets = pd.read_csv(RAW / "Sales target.csv")

print("Raw rows:", len(orders_raw), len(details), len(targets))

orders = orders_raw.dropna(subset=["Order ID"]).copy()
orders["order_date"] = pd.to_datetime(orders["Order Date"], format="%d-%m-%Y")
orders["State"] = orders["State"].str.strip()

print(f"Valid orders: {len(orders)}")
print(f"Date range: {orders['order_date'].min().date()} -> {orders['order_date'].max().date()}")

In [ ]:
print("=== Null counts (raw orders) ===")
print(orders_raw.isnull().sum())

print("\n=== Categories ===")
print(details["Category"].value_counts())

print("\n=== Join check ===")
o_ids = set(orders["Order ID"])
d_ids = set(details["Order ID"])
print(f"Orders without details: {len(o_ids - d_ids)}")
print(f"Details without orders: {len(d_ids - o_ids)}")

In [ ]:
monthly_sales = (
    details.merge(orders[["Order ID", "order_date"]], on="Order ID")
    .assign(month=lambda d: d["order_date"].dt.to_period("M"))
    .groupby(["month", "Category"], as_index=False)["Amount"]
    .sum()
)
monthly_sales.head(10)

## Data Quality Decisions (ETL)

1. **Drop 60 blank trailing rows** in `List of Orders.csv` (null `Order ID`)
2. **Trim whitespace** on `State` (`Kerala ` -> `Kerala`)
3. **Parse dates** as `DD-MM-YYYY`
4. **Parse target month** from `MMM-YY` to first-of-month DATE
5. **Do not expose** `CustomerName` in UI — use state/category aggregates